# Chapitre 3 — Le Cloud pour la data

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Expliquer** les avantages du cloud computing et identifier quand l'utiliser plutôt qu'un environnement local
2. **Configurer** un compte AWS et gérer les credentials de manière sécurisée
3. **Lire et écrire** des données depuis/vers AWS S3 en utilisant boto3 et pandas
4. **Comparer** les équivalences entre AWS, GCP et Azure pour le stockage de données

---


## 3.6 Écrire vers S3 avec Python

---

### Introduction : Maîtriser le flux de données vers S3

**Où se trouve votre donnée au moment où vous voulez l'envoyer ?**

Dans le développement Cloud, et particulièrement avec Python, nous manipulons la donnée sous deux formes :

***1. La donnée "Figée" (Le Disque)***

C'est le fichier que vous voyez dans votre explorateur de fichiers (un `.csv`, une image `.jpg`). Il est écrit physiquement sur votre disque dur (SSD/HDD).

* **L'enjeu :** On veut simplement le "copier-coller" vers le Cloud.
* **Le risque :** C'est une opération lente (I/O) et elle nécessite de l'espace de stockage sur la machine qui exécute le code.

***2. La donnée "Vivante" (La RAM)***

C'est la donnée qui n'existe que pendant que votre script tourne. C'est votre **DataFrame Pandas**, votre dictionnaire Python ou le résultat d'un calcul. Elle est stockée dans la mémoire vive (RAM).

* **L'enjeu :** On veut l'envoyer directement sur S3 sans passer par l'étape lente et inutile de créer un fichier temporaire sur l'ordinateur.
* **Le risque :** La RAM est limitée et coûteuse.

---

***Ce que nous allons apprendre :***

Dans ce cours, nous allons voir comment utiliser les trois outils majeurs du marché selon ces deux scénarios :

1. **Boto3 :** Pour comprendre les fondations (Buffers, Clients, Uploads).
2. **Pandas & s3fs :** Pour la simplicité d'écriture directe.
3. **AWS Wrangler :** Pour passer à l'échelle industrielle (Data Lakes et partitionnement).

---

### Outil 1 : Boto3

In [ ]:
!pip3 install boto3

In [ ]:
import boto3
from io import StringIO

# Créer le client S3
nom_s3_client_personlise = boto3.client('s3')

# Action : Initialisation du client S3.
# Explication : On demande à la bibliothèque boto3 de créer une interface de bas niveau avec le service S3. 
# Le "client" est l'outil qui va nous permettre d'appeler les API officielles d'AWS (comme put_object, list_objects, etc.).

/Users/safae/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


---

#### Uploader un fichier physique

C'est comme envoyer une lettre papier par la poste.

- Le fichier existe physiquement sur ton disque dur (ton ordinateur, un serveur EC2, ou un disque dur externe).

- Le flux : Ton code dit à AWS : "Va chercher ce fichier qui s'appelle ventes.csv à tel endroit sur mon disque et copie-le sur S3".

- Usage type : Tu as téléchargé un dataset, ou un logiciel a généré un rapport .log sur ton serveur.



In [ ]:
# Upload un fichier local
nom_s3_client_personlise.upload_file(
     Filename="ai_benchmark_dataset.csv",
     Bucket="mon-bucket",  # Remplacez par votre bucket
     Key="boto3_fichier_physique/fichier1.csv"
 )

print("boto3.upload_file() upload un fichier local vers S3")

# Mécanisme : Boto3 ouvre le fichier, le lit par morceaux (chunks) et l'envoie vers S3.
# Avantages :
# - Gestion automatique du "Multi-part upload" : Si le fichier est volumineux (plus de quelques Go), Boto3 le découpe automatiquement pour optimiser le transfert.
# - Simplicité : Pas besoin de gérer des flux de données ou de buffers.
# Inconvénient : Nécessite que le fichier soit écrit sur le disque au préalable.

boto3.upload_file() upload un fichier local vers S3


---

#### Avec buffer mémoire

C'est comme envoyer un e-mail que tu es en train de taper.

- Les données n'existent que dans ton script Python (dans une variable ou un DataFrame Pandas). Si tu coupes le courant à ce moment-là, les données sont perdues car elles n'ont jamais été enregistrées sous forme de fichier sur ton ordi.

- Le flux : Ton code dit à AWS : "Prends ce que j'ai actuellement dans ma mémoire vive et crée un objet directement sur S3 avec".

- Usage type : Tu as nettoyé des données avec Pandas, tu as le résultat final "en main" (dans la RAM), et tu veux l'envoyer sur S3 sans perdre de temps à l'écrire sur ton disque dur d'abord.

In [3]:
import pandas as pd

# Votre DataFrame nettoyé
df_clean = pd.DataFrame({
    'années': ['2026', '2025', '2024', '2026', '2025', '2024'],
    'ventes': [100, 200, 150, 300, 400, 650]
})
print(df_clean)

  années  ventes
0   2026     100
1   2025     200
2   2024     150
3   2026     300
4   2025     400
5   2024     650


In [ ]:
# Convertir le DataFrame en CSV dans un buffer mémoire

csv_buffer = StringIO()
# Action : Création d'un flux de texte en mémoire vive (RAM).
# Explication : StringIO simule un fichier texte, mais sans écrire sur le disque dur. 
# C'est comme si on ouvrait un bloc-notes invisible qui n'existe que dans la mémoire de l'ordinateur. C'est la clé pour éviter de créer des fichiers temporaires "polluants".

df_clean.to_csv(csv_buffer, index=False)
# Habituellement, on écrit df.to_csv("nom_du_fichier.csv"). Ici, on remplace le nom du fichier par notre variable csv_buffer.
# Pandas transforme les données en texte CSV.
# Au lieu de les envoyer vers le disque dur, il les "injecte" dans notre buffer en mémoire vive.
# index=False : On demande à Pandas de ne pas inclure la colonne des index (0, 1, 2...) dans le CSV final.

# Uploader vers S3
# C'est ici que le transfert vers le Cloud commence. Contrairement à upload_file, put_object est une méthode qui prend directement des données brutes (du texte ou des octets).
nom_s3_client_personlise.put_object(
    Bucket='mon-bucket',
    Key='boto3_buffer/fichier2.csv', # Dans S3, on ne parle pas de "dossiers" mais de "Clés" (Keys). Ici, le fichier s'appellera ventes_clean.csv et sera virtuellement placé dans un dossier nommé processed.
    Body=csv_buffer.getvalue() # dit à Python : "Récupère tout le texte que Pandas a écrit dans le buffer invisible et donne-le à AWS". C'est ce contenu qui deviendra le corps du fichier sur S3.
)

print("Upload réussi !")

# Éphémère : Idéal pour le cloud computing (Lambda, Fargate) car on ne laisse aucune trace de fichier local.
# Inconvénient : Attention à la RAM. Si ton DataFrame pèse 8 Go et que ta machine n'a que 4 Go de RAM, ton script va planter (Memory Error).

Performance et Coût : Écrire sur un disque (I/O) est l'opération la plus lente en informatique. Passer par la mémoire (put_object) est beaucoup plus rapide et économise des cycles de calcul (donc de l'argent).

En résumé :

**upload_file** = Source = Fichier déjà enregistré.

**put_object** = Source = Données "vivantes" dans ton code.

**Avantage** : Contrôle total, fonctionne partout où boto3 est installé.

**Quand l'utiliser** : Environnements avec contraintes de dépendances, ou quand vous avez besoin d'options avancées (encryption, metadata).

---

### Outil 2 : AWS SDK for pandas (awswrangler)

La solution "enterprise-grade" recommandée par AWS :

Le "plus" : Tu n'as pas besoin de créer de client S3, ni de gérer de buffer. Tu donnes ton DataFrame et l'adresse S3 (format URI), et Wrangler s'occupe de tout techniquement en arrière-plan.

C'est l'outil de Boto3 + Pandas sous stéroïdes


In [4]:
!pip3 install awswrangler

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 380 kB 7.3 MB/s eta 0:00:01
     |████████████████████████████████| 14.6 MB 113.0 MB/s eta 0:00:01
  Attempting uninstall: botocore
    Found existing installation: botocore 1.41.5
    Uninstalling botocore-1.41.5:
      Successfully uninstalled botocore-1.41.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.26.0 requires botocore<1.41.6,>=1.41.0, but you have botocore 1.42.33 which is incompatible.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [5]:
import awswrangler as wr

In [ ]:
# Uploader un fichier local (Physique) vers S3


wr.s3.upload(local_file="ai_benchmark_dataset.csv", path="s3://mon-bucket/wrangler_physique/fichier3.pdf")

In [ ]:
# Uploader un dataframe en mémoire

# Écriture simple
wr.s3.to_csv(df_clean, "s3://mon-bucket/wrangler_memoire/fichier4.csv", index=False) # adresse S3 format URI
# Exactement la même chose que le code précédent avec le buffer (StringIO) et put_object, mais en une seule ligne.

In [ ]:
# Avec partitionnement (pour les gros datasets)
wr.s3.to_csv(
    df_clean,
    "s3://mon-bucket/wrangler_memoire_partition/fichier5/",
    index=False,
    dataset=True,
    partition_cols=['annee']
)

# dataset=True : On ne crée pas un simple fichier, mais un ensemble de données organisé.
# partition_cols : Wrangler va découper ton gros DataFrame en plein de petits fichiers classés par dossiers.
# Exemple : S3 va créer une structure comme celle-ci :
#   processed/ventes/annee=2023/mois=01/file1.csv
#   processed/ventes/annee=2023/mois=02/file2.csv

**Avantage** : Fonctionnalités avancées (partitionnement, intégration Glue Catalog, gestion automatique des types).

**À quel moment est-ce utile ?**
- Cas n°1 : La productivité (Écriture simple)
C'est utile dès que tu veux écrire du code propre et court. Au lieu de 10 lignes de Boto3, tu en as une seule. C'est plus facile à lire et à maintenir pour une équipe.

- Cas n°2 : Les gros volumes de données (Partitionnement)
C'est le cas le plus important. On utilise le partitionnement quand :
Tu as beaucoup de données : Au lieu d'avoir un fichier ventes.csv de 10 Go (impossible à ouvrir), tu as 100 fichiers de 100 Mo.

**Installation** : `pip install awswrangler`

*(Source : [AWS SDK for pandas Documentation](https://aws-sdk-pandas.readthedocs.io/en/stable/tutorials/003%20-%20Amazon%20S3.html))*

### Outil 3 : Pandas natif avec URI S3

La plus simple — pandas gère tout en arrière-plan :

In [ ]:
!pip3 install s3fs
!pip3 install pyarrow
!pip3 install --upgrade s3fs pyarrow

In [ ]:
# Écriture CSV vers S3 
df_clean.to_csv("s3://mon-bucket/pandas_csv/fichier6.csv", index=False)

#mon_bucket = "demo-aws-simlpon"

print("DataFrame exemple :")
df_clean.head()

In [ ]:
# Écriture Parquet vers S3 (recommandé)
df_clean.to_parquet("s3://demo-aws-simlpon/pandas_parquet/fichier7.parquet")

# Ici, Pandas ne parle pas directement à S3. Il utilise la bibliothèque s3fs pour simuler un système de fichiers et pyarrow pour compresser la donnée au format Parquet. 
# C'est la méthode la plus simple pour passer de la donnée analysée au stockage Cloud.

**Avantage** : Simplicité maximale, une seule ligne de code.

**Limitation** : Nécessite la librairie `s3fs` installée, qui peut créer des conflits de dépendances dans certains environnements.

### Comparaison

| Méthode                | Type de donnée                                 | Formats supportés                     | Utilité principale                | Avantage clé                                             | À quel moment l'utiliser ?                          |
|------------------------|-----------------------------------------------|---------------------------------------|-----------------------------------|----------------------------------------------------------|----------------------------------------------------|
| Boto3                  | Physique (upload_file) & Mémoire (put_object) | Tous (CSV, JSON, Images, Zip...)      | Interaction native avec l'API AWS | Léger & Sans dépendance : Pas besoin d'installer de grosses librairies | AWS Lambda ou scripts de transfert simples          |
| AWS Wrangler           | Physique (upload) & Mémoire (to_csv, to_parquet) | CSV, Parquet, JSON, Excel...          | Data Engineering de niveau entreprise | Puissant & Intelligent : Gère seul le partitionnement et le Glue Catalog | Data Lakes, gros volumes de données et pipelines complexes |
| Pandas (+ s3fs)        | Mémoire uniquement (via DataFrame)            | CSV, Parquet, JSON                    | Analyse de données rapide          | Simplicité maximale : Une seule ligne de code avec l'URI s3:// | Notebooks (SageMaker/Jupyter) et prototypage rapide |

###  ✅ Le workflow

| Étape de la Pipeline       | État de la donnée                          | Méthode recommandée            | Format de fichier         | Dossier S3 Cible            | Pourquoi ce choix ?                                                                                               |
|----------------------------|-------------------------------------------|--------------------------------|---------------------------|-----------------------------|-------------------------------------------------------------------------------------------------------------------|
| 1. Collecte (Ingestion)    | Physique (Fichier brut externe)           | boto3.upload_file()            | Source (CSV, JSON, Logs)  | s3://bucket/raw/            | On ne modifie rien. On déplace le fichier tel quel pour garder une trace de l'original.                           |
| 2. Nettoyage (Processing)  | Elle vit dans la mémoire (DataFrame en cours de calcul)    | Buffer + put_object() ou wr.s3.to_csv() | CSV ou Parquet            | s3://bucket/processed/      | Évite les fichiers temporaires sur le disque. Le format Parquet commence à être intéressant ici pour gagner de la place. |
| 3. Rendu Final (Curated)   | Dataset organisé (Prêt pour analyse)      | awswrangler.s3.to_parquet()    | Parquet (Partitionné)     | s3://bucket/gold/ ou analytics/ | Optimise les coûts et la vitesse pour Athena. Le partitionnement facilite les requêtes sur de gros volumes.           |

**💡 Le schéma de décision**

| La donnée est-elle déjà sur le disque ? | Méthode recommandée | Pourquoi ? |
| --- | --- | --- |
| **OUI** (ex: `image.jpg` ou `data.csv`) | `upload_file()` | Simple et gère les gros volumes automatiquement. |
| **NON** (ex: un DataFrame Pandas après un calcul) | **Buffer + `put_object()**` | Évite les fichiers temporaires inutiles et gagne du temps. |

---

### Bonnes pratiques d'écriture

| Pratique | Raison |
|----------|--------|
| Utiliser Parquet en production | 3-10x plus compact que CSV. C'est le standard pour les données nettoyées et finales.|
| Organiser en dossiers par date | Utiliser des dossiers par date (ex: annee=2024/mois=01/) avec AWS Wrangler permet de ne lire que les données nécessaires et de réduire drastiquement la facture AWS. |
| Ne jamais écraser raw/ | Les données brutes sont votre "assurance vie" en cas d'erreur dans votre code de nettoyage. |
| Ajouter des métadonnées | Traçabilité (qui, quand, comment) |

